In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import torch

import os

from datasets.hyspecnet11k import HySpecNet11k
from metrics import psnr, sa

from models import _jpeg2000

In [ ]:
dataset_path = "./datasets/hyspecnet-11k/"
mode = "easy"

jp2k_path = "/tmp/tmp.jp2"

In [ ]:
test_dataset = HySpecNet11k(dataset_path, mode=mode, split="test", transform=None)

psnr_metric = psnr.PeakSignalToNoiseRatio()
sa_metric = sa.SpectralAngle()

In [ ]:
for cratio in [4, 8, 16, 32, 64, 128, 256, 512, 1024]:
    cr_list = []
    psnr_list = []
    sa_list = []

    model = _jpeg2000.JPEG2000(target_compression_ratio=cratio, rescale=True)

    for data_org in test_dataset:
        data_org = data_org.unsqueeze(0)
        data_rec = model(data_org)

        size_raw = torch.Tensor.numpy(data_org).nbytes

        size_jp2k = os.path.getsize(jp2k_path)

        cr = size_raw / size_jp2k

        psnr = psnr_metric(data_org, data_rec)

        sa = sa_metric(data_org, data_rec)

        cr_list.append(cr)
        psnr_list.append(psnr)
        sa_list.append(sa)

    cr_avg = np.mean(cr_list)
    psnr_avg = np.mean(psnr_list)
    sa_avg = np.nanmean(sa_list)

    print(f"cr_avg:\t\t{cr_avg}")
    print(f"psnr_avg:\t{psnr_avg} dB")
    print(f"sa_avg: \t{sa_avg} °")
    print("===")